Obtención de matrículas a partir del dataset (ejecutar solo una vez)

In [ ]:
import json
import cv2
import numpy as np
from pathlib import Path

DATASET = Path("dataset")
OUT = Path("plates")
MIN_W = 60
PAD = 0.05
IMG_EXT = {".jpg", ".jpeg", ".png"}

def leer(p):
    return cv2.imdecode(np.fromfile(str(p), np.uint8), cv2.IMREAD_COLOR)

def guardar(p, img):
    cv2.imencode(".jpg", img)[1].tofile(str(p))

for split in ["train", "val", "test"]:
    (OUT / split).mkdir(parents=True, exist_ok=True)

    imagenes = {p.stem: p for p in (DATASET / split / "images").iterdir()
                if p.suffix.lower() in IMG_EXT}
    jsons = {p.stem: p for p in (DATASET / split / "labels").glob("*.json")}
    parejas = sorted(set(imagenes) & set(jsons))

    validas = []   # (nombre, ruta_imagen, caja, texto)
    for nombre in parejas:
        for i, obj in enumerate(json.load(open(jsons[nombre], encoding="utf-8"))):
            texto = str(obj.get("plate") or "").replace(" ", "").upper()
            if texto and texto != "NONE":
                validas.append((f"{nombre}_{i}", imagenes[nombre], obj["rectangle"], texto))

    lineas, pequenas = [], 0
    for nombre, ruta, (cx, cy, w, h), texto in validas:
        img = leer(ruta)
        H, W = img.shape[:2]
        x1 = max(int((cx - w / 2 - w * PAD) * W), 0)
        x2 = min(int((cx + w / 2 + w * PAD) * W), W)
        y1 = max(int((cy - h / 2 - h * PAD) * H), 0)
        y2 = min(int((cy + h / 2 + h * PAD) * H), H)
        if x2 - x1 < MIN_W:
            pequenas += 1
            continue
        guardar(OUT / split / f"{nombre}.jpg", img[y1:y2, x1:x2])
        lineas.append(f"{split}/{nombre}.jpg\t{texto}")

    (OUT / f"{split}.txt").write_text("\n".join(lineas), encoding="utf-8")
    print(f"{split}: imágenes {len(imagenes)}, jsons {len(jsons)}, parejas {len(parejas)}, "
          f"con matrícula {len(validas)}, descartadas por ancho {pequenas}, "
          f"finales {len(lineas)}")

Importación de paquetes

In [1]:
import os
import subprocess
import sys

from paddleocr import PaddleOCR
import matplotlib.pyplot as plt
import matplotlib.patches as patches

c:\Users\Dylan\anaconda3\envs\P4_paddle\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Entrenamiento lector de matrículas

In [ ]:
DATA_DIR = "plates"
EPOCHS = 50
BATCH_SIZE = 8
LR = 1e-4
OUT_DIR = "results/plate-ocr"
CONFIG = "PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_mobile_rec.yml"
PRETRAINED = ("https://paddle-model-ecology.bj.bcebos.com/paddlex/"
              "official_pretrained_model/PP-OCRv5_mobile_rec_pretrained.pdparams")

def correr(cmd):
    """Ejecuta el comando y muestra su salida (stdout+stderr) en el notebook."""
    prefix = os.path.dirname(sys.executable)   # carpeta del entorno VC_P4
    env = os.environ.copy()
    env["PATH"] = os.pathsep.join([
        prefix,
        os.path.join(prefix, "Library", "bin"),
        os.path.join(prefix, "Scripts"),
        env["PATH"],
    ])
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding="utf-8", errors="replace", env=env)
    for linea in p.stdout:
        print(linea, end="")
    if p.wait() != 0:
        raise RuntimeError(f"El comando terminó con código {p.returncode} (mira la salida de arriba)")

# Diccionario de caracteres (matrículas: dígitos y letras)
DICT_PATH = f"{DATA_DIR}/dict.txt"
with open(DICT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"))

# Clonar PaddleOCR (solo la primera vez)
if not os.path.exists("PaddleOCR"):
    correr(["git", "clone", "--depth", "1", "https://github.com/PaddlePaddle/PaddleOCR.git"])
    correr([sys.executable, "-m", "pip", "install", "-r", "PaddleOCR/requirements.txt"])

# Entrenar
overrides = [
    f"Global.epoch_num={EPOCHS}",
    f"Global.save_model_dir={OUT_DIR}",
    f"Global.pretrained_model={PRETRAINED}",
    f"Global.character_dict_path={DICT_PATH}",
    "Global.use_space_char=False",
    "Global.eval_batch_step=[0,100]",
    f"Optimizer.lr.learning_rate={LR}",
    f"Train.dataset.data_dir={DATA_DIR}",
    f"Train.dataset.label_file_list=['{DATA_DIR}/train.txt']",
    f"Train.loader.batch_size_per_card={BATCH_SIZE}",
    "Train.loader.num_workers=0",
    f"Eval.dataset.data_dir={DATA_DIR}",
    f"Eval.dataset.label_file_list=['{DATA_DIR}/val.txt']",
    f"Eval.loader.batch_size_per_card={BATCH_SIZE}",
    "Eval.loader.num_workers=0",
]
correr([sys.executable, "-u", "PaddleOCR/tools/train.py", "-c", CONFIG, "-o", *overrides])

# Exportar el mejor modelo
correr([sys.executable, "PaddleOCR/tools/export_model.py", "-c", CONFIG, "-o",
        f"Global.pretrained_model={OUT_DIR}/best_accuracy",
        f"Global.save_inference_dir={OUT_DIR}/inference",
        f"Global.character_dict_path={DICT_PATH}"])

print("Listo. Modelo en", f"{OUT_DIR}/inference")

Skipping import of the encryption module.
[2026/11/09 22:12:04] ppocr INFO: Architecture : 
[2026/11/09 22:12:04] ppocr INFO:     Backbone : 
[2026/11/09 22:12:04] ppocr INFO:         name : PPLCNetV3
[2026/11/09 22:12:04] ppocr INFO:         scale : 0.95
[2026/11/09 22:12:04] ppocr INFO:     Head : 
[2026/11/09 22:12:04] ppocr INFO:         head_list : 
[2026/11/09 22:12:04] ppocr INFO:             CTCHead : 
[2026/11/09 22:12:04] ppocr INFO:                 Head : 
[2026/11/09 22:12:04] ppocr INFO:                     fc_decay : 1e-05
[2026/11/09 22:12:04] ppocr INFO:                 Neck : 
[2026/11/09 22:12:04] ppocr INFO:                     depth : 2
[2026/11/09 22:12:04] ppocr INFO:                     dims : 120
[2026/11/09 22:12:04] ppocr INFO:                     hidden_dims : 120
[2026/11/09 22:12:04] ppocr INFO:                     kernel_size : [1, 3]
[2026/11/09 22:12:04] ppocr INFO:                     name : svtr
[2026/11/09 22:12:04] ppocr INFO:                     use

RuntimeError: El comando terminó con código 1 (mira la salida de arriba)